# Phase 21 — probabilistic multiscale regime models

This notebook compares contemporaneous clustering, an independent Gaussian mixture, and causal-filtered HMMs against the frozen direct taxonomy. The direct taxonomy is a comparator, not a supervised target or hidden truth.

In [1]:
import os

# Reproducible HMM fits (M15): fix the BLAS/OpenMP and PyTorch thread counts before the numerical
# libraries load. A concurrent or differently threaded run can change floating-point reduction order
# and therefore the EM path. Restart the kernel after changing HMM_THREADS.
HMM_THREADS = 1
for _thread_variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_thread_variable, str(HMM_THREADS))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import adjusted_rand_score

from dc_basics.data import resolve_data_root
from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    build_multiscale_state,
    candidate_summary,
    emission_feature_contributions,
    filtering_prefix_invariant,
    fit_candidate,
    load_close,
    plot_price_states,
    prepare_data,
    ratio_ladder,
    semantic_state_name,
)

torch.set_num_threads(HMM_THREADS)

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
PRIMARY_CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
N_STATES = 4
RANDOM_STATE = 17
MAX_ITER = 40
N_RESTARTS = 2
MIN_EMISSION_VARIANCE = 1e-2
DISPLAY_FAMILY = ModelFamily.NORMAL_HMM
RUN_BAR_TIME_HMM_SENSITIVITY = False
STABILITY_SEEDS = (29, 47)
LAST_N = 10_000

## Frozen comparison contract

All models use four states for this first comparison because the direct baseline has four structural classes. Every candidate receives the same **continuous-only**, train-scaled causal feature matrix. Features that are training-constant, nearly discrete, or dominated by one repeated value (point-mass share of at least 20%, for example mode age 0 on the scale that just fired) are screened out on training rows, because a continuous density would isolate that value at its variance floor. Discrete mode geometry, fragmentation, direction, and firing counts remain direct-baseline or diagnostic fields; they are not treated as Gaussian or Student-t observations.

State semantics are mapped one-to-one using the training partition only. A surplus or unsupported state remains `unmatched`; it is never assigned a convenient duplicate name. Agreement with the direct taxonomy measures relationship, not accuracy against truth.

- K-means: contemporaneous hard clustering; no temporal state.
- Gaussian mixture: contemporaneous soft membership; no Markov history.
- Normal HMM: Gaussian emissions plus filtered Markov history.
- Student-t HMM: heavier-tailed emissions plus filtered Markov history.

The primary clock is union-event time. A Normal-HMM bar-time sensitivity remains available as an explicit opt-in because changing clocks changes the population and transition meaning. Under the frozen full-history settings, the reference run exceeded both 30- and 60-minute cell limits; disabling it by default records an operational limit, not a favorable or unfavorable model result. Variance-floor hits are reported because high confidence can still be numerical rather than economic.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
_, global_state, _ = build_multiscale_state(prices, THRESHOLDS, include_scale_state=False)
prepared = prepare_data(global_state, PRIMARY_CLOCK, FEATURE_TIER)

split_summary = pd.DataFrame.from_records([
    {"split": name, "rows": len(frame), "start": frame.index[0], "end": frame.index[-1]}
    for name, frame in prepared.frames.items()
]).set_index("split")
display(split_summary)
print(f"Retained features: {len(prepared.feature_columns)}")
print(f"Removed emission features and reasons: {prepared.removed_feature_reasons}")
print(f"Dropped rows: {prepared.dropped_rows}; by regime: {prepared.dropped_regime_counts}")
print(f"Train end: {prepared.split.train_end}; validation end: {prepared.split.validation_end}")

,rows,start,end
split,,,
train,3273,2023-05-25 09:35:00+02:00,2025-06-23 05:30:00+02:00
validation,938,2025-06-23 07:10:00+02:00,2026-03-03 06:15:00+02:00
test,851,2026-03-03 09:40:00+02:00,2026-11-09 10:15:00+02:00


Retained features: 17
Removed emission features and reasons: {'reversal_progress_00': 'training_constant_or_discrete', 'log1p_mode_age_00': 'point_mass_share_0.527', 'reversal_progress_01': 'point_mass_share_0.523', 'log1p_mode_age_01': 'point_mass_share_0.274', 'reversal_progress_02': 'point_mass_share_0.262'}
Dropped rows: 0; by regime: {'broad_coherence': 0, 'shallow_counter': 0, 'deep_counter': 0, 'fragmented': 0}
Train end: 2025-06-23 05:55:00+02:00; validation end: 2026-03-03 07:20:00+02:00


## Fit all predeclared candidates

Scaling is fitted on training rows only. HMM probabilities are forward-filtered. Validation and test state semantics use the immutable mapping learned from the training contingency table.

In [3]:
candidate_families = (
    ModelFamily.KMEANS,
    ModelFamily.GAUSSIAN_MIXTURE,
    ModelFamily.NORMAL_HMM,
    ModelFamily.STUDENTT_HMM,
)
fits = {}
summary_records = []
for family in candidate_families:
    fit = fit_candidate(
        prepared, family, N_STATES, RANDOM_STATE, max_iter=MAX_ITER, n_restarts=N_RESTARTS,
        min_emission_variance=MIN_EMISSION_VARIANCE,
    )
    fits[family] = fit
    record = candidate_summary(fit, prepared)
    record["filtering_prefix_invariant"] = filtering_prefix_invariant(fit, prepared)
    summary_records.append(record)

candidate_table = pd.DataFrame.from_records(summary_records).set_index("family")
display(candidate_table.style.format({
    "validation_log_likelihood_per_row": "{:.4f}",
    "minimum_state_occupancy": "{:.2%}",
    "switch_rate": "{:.2%}",
    "median_dwell": "{:.1f}",
    "mean_max_probability": "{:.2%}",
    "baseline_adjusted_rand": "{:.3f}",
    "baseline_nmi": "{:.3f}",
    "semantic_mapping_coverage": "{:.2%}",
    "baseline_mapped_agreement": "{:.2%}",
    "variance_floor_hit_share": "{:.2%}",
}))

,n_states,clock,tier,feature_count,removed_feature_count,prepared_rows_dropped,error,validation_log_likelihood_per_row,minimum_state_occupancy,switch_rate,median_dwell,mean_max_probability,baseline_adjusted_rand,baseline_nmi,semantic_mapping_coverage,baseline_mapped_agreement,training_classes_without_state,minimum_fitted_variance,variance_floor_hit_count,variance_floor_hit_share,filtering_prefix_invariant
family,,,,,,,,,,,,,,,,,,,,,
kmeans,4,union_event_time,dynamic,17,5,0,None,nan,14.82%,49.41%,1.0,33.48%,0.357,0.435,100.00%,70.26%,(),nan,0,nan%,None
gaussian_mixture,4,union_event_time,dynamic,17,5,0,None,-19.3030,15.67%,59.12%,1.0,96.38%,0.433,0.540,100.00%,75.48%,(),0.081092,0,0.00%,None
normal_hmm,4,union_event_time,dynamic,17,5,0,None,-18.9105,15.67%,55.92%,1.0,97.50%,0.510,0.586,100.00%,79.64%,(),0.065909,0,0.00%,True
studentt_hmm,4,union_event_time,dynamic,17,5,0,None,-20.2688,14.82%,53.68%,1.0,97.24%,0.557,0.601,100.00%,81.77%,(),0.067097,0,0.00%,True


## State occupancy and training-only semantic maps

A model state mapped to `fragmented` means that its training observations overlapped that direct class most strongly under the one-to-one assignment. The model state remains its own probabilistic object. A partition is a stable object to name only if it passes the restart-stability check further down; until then these annotations are provisional.

In [4]:
map_records = []
occupancy_records = []
for family, fit in fits.items():
    if fit.error is not None:
        continue
    for state, regime_code in fit.semantic_map.items():
        map_records.append({
            "family": family.value,
            "raw_state": state,
            "training_semantic_map": semantic_state_name(regime_code),
        })
    for split_name, states in fit.states.items():
        counts = np.bincount(states, minlength=N_STATES) / len(states)
        for state, share in enumerate(counts):
            occupancy_records.append({
                "family": family.value, "split": split_name, "state": state, "share": share
            })
display(pd.DataFrame.from_records(map_records).set_index(["family", "raw_state"]))
occupancy = pd.DataFrame.from_records(occupancy_records).pivot_table(
    index=["family", "state"], columns="split", values="share"
)
display(occupancy.style.format("{:.2%}"))

training_semantic_map
family           raw_state                      
kmeans           0               shallow_counter
                 1                    fragmented
                 2                  deep_counter
                 3               broad_coherence
gaussian_mixture 0               shallow_counter
                 1                    fragmented
                 2                  deep_counter
                 3               broad_coherence
normal_hmm       0               broad_coherence
                 1               shallow_counter
                 2                  deep_counter
                 3                    fragmented
studentt_hmm     0               broad_coherence
                 1               shallow_counter
                 2                  deep_counter
                 3                    fragmented

## Restart stability (descriptive)

Raw state numbers are arbitrary, so each family is refitted with independent seeds and compared with the primary fit using the permutation-invariant adjusted Rand index on validation observations. This answers Chapter 25's restart-stability question. In this phase it is descriptive; Phase 22 turns a predeclared minimum restart ARI into an eligibility rule that every candidate must pass before likelihood ranking. The test partition is not used here.

In [5]:
stability_records = []
for family, fit in fits.items():
    for seed in STABILITY_SEEDS:
        refit = fit_candidate(
            prepared, family, N_STATES, seed, max_iter=MAX_ITER, n_restarts=N_RESTARTS,
            min_emission_variance=MIN_EMISSION_VARIANCE,
        )
        both_fitted = fit.error is None and refit.error is None
        stability_records.append({
            "family": family.value,
            "seed": seed,
            "error": refit.error,
            "validation_ari_vs_primary": (
                adjusted_rand_score(fit.states["validation"], refit.states["validation"])
                if both_fitted else np.nan
            ),
        })
restart_stability = pd.DataFrame.from_records(stability_records).set_index(["family", "seed"])
display(restart_stability.style.format({"validation_ari_vs_primary": "{:.3f}"}))
display(
    restart_stability.groupby("family")["validation_ari_vs_primary"].min()
    .rename("minimum_validation_ari_vs_primary").to_frame().style.format("{:.3f}")
)

,minimum_validation_ari_vs_primary
family,
gaussian_mixture,0.797
kmeans,0.984
normal_hmm,0.418
studentt_hmm,0.482


## Which features carry the density likelihood

Every density candidate uses diagonal covariances, so its validation log density splits exactly into one term per feature. `gain_vs_standard_normal` compares each feature's state-weighted log density with one standard normal on the same train-standardised values. A large gain concentrated in features whose states sit at the variance floor would mean the likelihood is buying itself by isolating repeated values rather than describing continuous variation.

In [6]:
for family, fit in fits.items():
    contributions = emission_feature_contributions(fit, prepared, "validation")
    if contributions.empty:
        continue
    floored = contributions.loc[contributions["states_at_variance_floor"].gt(0)]
    print(
        f"{family.value}: {len(floored)} feature(s) with a state at the variance floor; "
        f"largest validation likelihood gains:"
    )
    display(contributions.head(8).style.format({
        "weighted_log_density": "{:.3f}",
        "gain_vs_standard_normal": "{:.3f}",
        "training_point_mass_share": "{:.2%}",
        "share_of_total_gain": "{:.1%}",
    }))

gaussian_mixture: 0 feature(s) with a state at the variance floor; largest validation likelihood gains:


,weighted_log_density,gain_vs_standard_normal,states_at_variance_floor,training_point_mass_share,share_of_total_gain
feature,,,,,
max_reversal_progress,-0.221,1.181,0,5.01%,20.0%
mean_reversal_progress,-0.657,0.757,0,5.01%,12.8%
reversal_progress_04,-0.738,0.672,0,5.01%,11.4%
log1p_mode_age_02,-1.000,0.449,0,13.63%,7.6%
reversal_progress_03,-0.971,0.445,0,11.89%,7.5%
log1p_live_tmv_00,-0.927,0.436,0,0.06%,7.4%
log1p_live_tmv_02,-1.077,0.302,0,0.06%,5.1%
log1p_mode_age_03,-1.127,0.257,0,6.23%,4.4%


normal_hmm: 0 feature(s) with a state at the variance floor; largest validation likelihood gains:


,weighted_log_density,gain_vs_standard_normal,states_at_variance_floor,training_point_mass_share,share_of_total_gain
feature,,,,,
max_reversal_progress,-0.217,1.185,0,5.01%,20.0%
mean_reversal_progress,-0.664,0.750,0,5.01%,12.7%
reversal_progress_04,-0.774,0.636,0,5.01%,10.7%
log1p_mode_age_02,-0.978,0.471,0,13.63%,8.0%
reversal_progress_03,-0.981,0.436,0,11.89%,7.4%
log1p_live_tmv_00,-0.944,0.420,0,0.06%,7.1%
log1p_live_tmv_02,-1.007,0.372,0,0.06%,6.3%
log_live_r_02,-1.130,0.293,0,0.03%,4.9%


studentt_hmm: 0 feature(s) with a state at the variance floor; largest validation likelihood gains:


,weighted_log_density,gain_vs_standard_normal,states_at_variance_floor,training_point_mass_share,share_of_total_gain
feature,,,,,
max_reversal_progress,-0.355,1.047,0,5.01%,23.5%
mean_reversal_progress,-0.716,0.698,0,5.01%,15.7%
reversal_progress_04,-0.848,0.561,0,5.01%,12.6%
reversal_progress_03,-1.033,0.383,0,11.89%,8.6%
log1p_live_tmv_00,-0.988,0.375,0,0.06%,8.4%
log1p_mode_age_02,-1.082,0.367,0,13.63%,8.2%
log1p_live_tmv_02,-1.045,0.334,0,0.06%,7.5%
log1p_live_tmv_01,-1.269,0.238,0,0.06%,5.3%


## Observation-clock sensitivity

This is not a search for the clock with the most attractive labels. It asks how much persistence is introduced when the same evolving hierarchy is sampled on every bar instead of only on union-DCC bars.

In [7]:
clock_sensitivity = []
if RUN_BAR_TIME_HMM_SENSITIVITY:
    bar_prepared = prepare_data(global_state, ObservationClock.BAR, FEATURE_TIER, split=prepared.split)
    bar_fit = fit_candidate(
        bar_prepared, ModelFamily.NORMAL_HMM, N_STATES, RANDOM_STATE, max_iter=MAX_ITER,
        n_restarts=N_RESTARTS, min_emission_variance=MIN_EMISSION_VARIANCE,
    )
    event_record = candidate_summary(fits[ModelFamily.NORMAL_HMM], prepared)
    bar_record = candidate_summary(bar_fit, bar_prepared)
    event_record["filtering_prefix_invariant"] = filtering_prefix_invariant(
        fits[ModelFamily.NORMAL_HMM], prepared
    )
    bar_record["filtering_prefix_invariant"] = filtering_prefix_invariant(bar_fit, bar_prepared)
    clock_sensitivity = pd.DataFrame.from_records([event_record, bar_record]).set_index("clock")
    display(clock_sensitivity)
else:
    print(
        "Bar-time HMM sensitivity is opt-in: the frozen full-history fit exceeded the "
        "60-minute reference cell limit. No bar-time result is inferred."
    )

Bar-time HMM sensitivity is opt-in: the frozen full-history fit exceeded the 60-minute reference cell limit. No bar-time result is inferred.


## Recent filtered HMM states on price

The displayed state is post-observation and causal at each union-event timestamp. It is not smoothed backward. Visually similar observations may receive different states because the filtered prior probability enters the HMM decision. When `mean_max_probability` in the candidate table is close to 1, the emission term dominates and the Markov prior rarely changes the assigned state; read persistence claims with that in mind.

In [8]:
display_fit = fits[DISPLAY_FAMILY]
if display_fit.error is not None:
    raise RuntimeError(f"The predeclared display HMM failed: {display_fit.error}")
display_parts = []
for split_name in ("train", "validation", "test"):
    part = prepared.frames[split_name].copy()
    part["model_state"] = display_fit.states[split_name]
    part["model_state_label"] = [f"state_{state}" for state in display_fit.states[split_name]]
    part["training_overlap_annotation"] = [
        semantic_state_name(code) for code in display_fit.mapped_regimes[split_name]
    ]
    for state in range(display_fit.n_states):
        part[f"filtered_p_state_{state}"] = display_fit.probabilities[split_name][:, state]
    display_parts.append(part)
display_frame = pd.concat(display_parts).sort_index()
probability_columns = [f"filtered_p_state_{state}" for state in range(display_fit.n_states)]
plot_price_states(
    display_frame,
    "model_state_label",
    f"{SYMBOL}: predeclared {display_fit.family.value} raw causal filtered states",
    probability_columns=probability_columns,
    last_n=LAST_N,
).show()
display(pd.crosstab(
    display_frame["model_state_label"], display_frame["training_overlap_annotation"], normalize="index"
).style.format("{:.2%}"))

training_overlap_annotation,broad_coherence,deep_counter,fragmented,shallow_counter
model_state_label,,,,
state_0,100.00%,0.00%,0.00%,0.00%
state_1,0.00%,0.00%,0.00%,100.00%
state_2,0.00%,100.00%,0.00%,0.00%
state_3,0.00%,0.00%,100.00%,0.00%


# Phase 21 completion gate

1. K-means and the mixture classify the current observation; only the HMM carries filtered state history.
2. Filtered probabilities, not smoothed probabilities, are the deployable HMM output.
3. Training-only semantic mapping prevents validation/test relabelling, and unsupported states remain `unmatched`.
4. Raw state IDs remain primary; overlap names are annotations, not a redefinition of the fitted state.
5. Discrete hierarchy geometry is compared after fitting rather than supplied to a continuous emission density.
6. Agreement with the direct taxonomy is descriptive because the taxonomy is not truth.
7. High persistence or posterior confidence is not automatically useful; inspect variance-floor hits.
8. Bar-time and event-time HMMs model different transition processes.
9. Model selection is deferred to the predeclared causal/OOS phase.
10. Restart stability is read for every family before any selection; low adjusted-Rand agreement means the partition is not yet a stable object to name.
11. Per-feature likelihood contributions show whether a density model's advantage comes from continuous variation or from isolating repeated values at the variance floor.